# Malware Analysis Laboratory

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import base64
import re
import json
from datetime import datetime, timedelta
import random
from collections import defaultdict, Counter
import struct
import binascii
import warnings
warnings.filterwarnings('ignore')

class MalwareAnalysisLab:
    def __init__(self):
        self.sample_database = self.generate_sample_database()
        self.yara_rules = self.load_yara_rules()
        self.behavioral_patterns = self.load_behavioral_patterns()

    def generate_sample_database(self):
        np.random.seed(42)
        malware_families = ['Zeus', 'Emotet', 'TrickBot', 'Ryuk', 'Cobalt Strike', 'Mimikatz', 'PowerShell Empire', 'Metasploit']
        file_types = ['PE32', 'ELF', 'PDF', 'DOC', 'JS', 'VBS', 'BAT', 'PS1']

        samples = []
        for i in range(100):
            sample = {
                'id': f"MAL_{i+1:04d}",
                'family': np.random.choice(malware_families),
                'file_type': np.random.choice(file_types),
                'size': np.random.randint(1024, 10485760),
                'md5': hashlib.md5(f"sample_{i}".encode()).hexdigest(),
                'sha1': hashlib.sha1(f"sample_{i}".encode()).hexdigest(),
                'sha256': hashlib.sha256(f"sample_{i}".encode()).hexdigest(),
                'first_seen': datetime.now() - timedelta(days=np.random.randint(1, 1000)),
                'detection_ratio': f"{np.random.randint(15, 67)}/67",
                'entropy': np.random.uniform(3.0, 8.0),
                'packer': np.random.choice(['UPX', 'ASPack', 'PEtite', 'None'], p=[0.3, 0.2, 0.1, 0.4]),
                'capabilities': self.generate_capabilities()
            }
            samples.append(sample)
        return samples

    def generate_capabilities(self):
        all_caps = [
            'file_manipulation', 'registry_modification', 'network_communication',
            'process_injection', 'keylogging', 'screen_capture', 'cryptocurrency_mining',
            'ransomware_encryption', 'credential_theft', 'lateral_movement',
            'persistence_mechanism', 'evasion_technique', 'anti_analysis',
            'command_control', 'data_exfiltration'
        ]
        num_caps = np.random.randint(2, 8)
        return random.sample(all_caps, num_caps)

    def load_yara_rules(self):
        return {
            'zeus_banking_trojan': {
                'strings': ['$mz = { 4D 5A }', '$api1 = "CreateProcessA"', '$api2 = "VirtualAlloc"'],
                'condition': '$mz at 0 and $api1 and $api2',
                'description': 'Detects Zeus banking trojan variants'
            },
            'ransomware_generic': {
                'strings': ['$crypt1 = "CryptEncrypt"', '$crypt2 = "CryptGenKey"', '$ext = ".encrypted"'],
                'condition': 'any of ($crypt*) and $ext',
                'description': 'Generic ransomware detection'
            },
            'powershell_obfuscated': {
                'strings': ['$ps1 = "powershell"', '$b64 = /[A-Za-z0-9+\/]{50,}={0,2}/', '$invoke = "IEX"'],
                'condition': '$ps1 and $b64 and $invoke',
                'description': 'Obfuscated PowerShell script'
            },
            'credential_harvester': {
                'strings': ['$lsass = "lsass.exe"', '$sam = "SAM"', '$ntds = "ntds.dit"'],
                'condition': 'any of them',
                'description': 'Credential harvesting tools'
            }
        }

    def load_behavioral_patterns(self):
        return {
            'network_beaconing': {
                'description': 'Regular network communication pattern',
                'indicators': ['periodic_dns_queries', 'http_post_requests', 'encrypted_traffic'],
                'risk_score': 8.5
            },
            'file_encryption': {
                'description': 'Mass file encryption behavior',
                'indicators': ['file_rename_operations', 'crypto_api_calls', 'ransom_note_creation'],
                'risk_score': 9.8
            },
            'credential_dumping': {
                'description': 'Memory credential extraction',
                'indicators': ['lsass_process_access', 'registry_sam_access', 'mimikatz_strings'],
                'risk_score': 9.2
            },
            'lateral_movement': {
                'description': 'Network propagation attempt',
                'indicators': ['smb_enumeration', 'rdp_brute_force', 'wmi_execution'],
                'risk_score': 8.8
            }
        }

    def static_analysis(self, sample_id):
        sample = next((s for s in self.sample_database if s['id'] == sample_id), None)
        if not sample:
            return {"error": "Sample not found"}

        analysis = {
            'sample_id': sample_id,
            'file_info': {
                'family': sample['family'],
                'type': sample['file_type'],
                'size': sample['size'],
                'entropy': round(sample['entropy'], 2)
            },
            'hashes': {
                'md5': sample['md5'],
                'sha1': sample['sha1'],
                'sha256': sample['sha256']
            },
            'packer': sample['packer'],
            'strings_analysis': self.extract_strings(sample),
            'pe_analysis': self.pe_analysis(sample) if sample['file_type'] == 'PE32' else None,
            'yara_matches': self.yara_scan(sample)
        }

        return analysis

    def extract_strings(self, sample):
        suspicious_strings = [
            'CreateRemoteThread', 'VirtualAllocEx', 'WriteProcessMemory',
            'GetProcAddress', 'LoadLibrary', 'SetWindowsHookEx',
            'keylogger', 'password', 'bitcoin', 'wallet.dat',
            'cmd.exe', 'powershell.exe', 'wscript.exe'
        ]

        found_strings = random.sample(suspicious_strings, np.random.randint(3, 8))

        return {
            'total_strings': np.random.randint(50, 500),
            'suspicious_strings': found_strings,
            'base64_encoded': np.random.randint(0, 10),
            'urls': [f"http://malicious-c2-{i}.com" for i in range(np.random.randint(1, 4))]
        }

    def pe_analysis(self, sample):
        return {
            'compilation_timestamp': (datetime.now() - timedelta(days=np.random.randint(1, 365))).strftime('%Y-%m-%d'),
            'sections': [
                {'name': '.text', 'virtual_size': '0x1000', 'entropy': 6.2},
                {'name': '.data', 'virtual_size': '0x800', 'entropy': 4.1},
                {'name': '.rsrc', 'virtual_size': '0x600', 'entropy': 7.8}
            ],
            'imports': [
                'kernel32.dll', 'user32.dll', 'advapi32.dll', 'ws2_32.dll'
            ],
            'exports': np.random.randint(0, 20),
            'digital_signature': np.random.choice(['Valid', 'Invalid', 'None'], p=[0.1, 0.2, 0.7])
        }

    def yara_scan(self, sample):
        matches = []
        for rule_name, rule_data in self.yara_rules.items():
            if np.random.random() < 0.3:
                matches.append({
                    'rule': rule_name,
                    'description': rule_data['description'],
                    'confidence': np.random.uniform(0.6, 1.0)
                })
        return matches

    def dynamic_analysis(self, sample_id):
        sample = next((s for s in self.sample_database if s['id'] == sample_id), None)
        if not sample:
            return {"error": "Sample not found"}

        analysis = {
            'sample_id': sample_id,
            'execution_time': np.random.randint(30, 300),
            'process_tree': self.generate_process_tree(),
            'network_activity': self.generate_network_activity(),
            'file_operations': self.generate_file_operations(),
            'registry_operations': self.generate_registry_operations(),
            'behavioral_analysis': self.behavioral_analysis(sample)
        }

        return analysis

    def generate_process_tree(self):
        processes = [
            {'pid': 1234, 'name': 'malware.exe', 'parent': 0, 'cmdline': 'C:\\temp\\malware.exe'},
            {'pid': 1235, 'name': 'cmd.exe', 'parent': 1234, 'cmdline': 'cmd.exe /c whoami'},
            {'pid': 1236, 'name': 'powershell.exe', 'parent': 1234, 'cmdline': 'powershell.exe -enc ...'}
        ]
        return processes

    def generate_network_activity(self):
        return {
            'dns_queries': [
                {'domain': 'malicious-c2.com', 'type': 'A', 'response': '192.168.1.100'},
                {'domain': 'evil-domain.net', 'type': 'TXT', 'response': 'encrypted_command'}
            ],
            'http_requests': [
                {'url': 'http://malicious-c2.com/gate.php', 'method': 'POST', 'status': 200},
                {'url': 'http://evil-domain.net/data', 'method': 'GET', 'status': 404}
            ],
            'tcp_connections': [
                {'dst_ip': '192.168.1.100', 'dst_port': 443, 'state': 'ESTABLISHED'},
                {'dst_ip': '10.0.0.50', 'dst_port': 445, 'state': 'SYN_SENT'}
            ]
        }

    def generate_file_operations(self):
        return [
            {'operation': 'CREATE', 'path': 'C:\\temp\\backdoor.exe', 'size': 2048},
            {'operation': 'DELETE', 'path': 'C:\\Windows\\System32\\logs\\security.log', 'size': 0},
            {'operation': 'MODIFY', 'path': 'C:\\Users\\victim\\Documents\\important.docx', 'size': 1024}
        ]

    def generate_registry_operations(self):
        return [
            {'operation': 'SET', 'key': 'HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run', 'value': 'Malware'},
            {'operation': 'CREATE', 'key': 'HKCU\\Software\\Malware\\Config', 'value': 'encrypted_config'},
            {'operation': 'DELETE', 'key': 'HKLM\\SYSTEM\\ControlSet001\\Services\\Defender', 'value': ''}
        ]

    def behavioral_analysis(self, sample):
        detected_behaviors = []

        for behavior_name, behavior_data in self.behavioral_patterns.items():
            if sample['family'] in ['Zeus', 'Emotet', 'TrickBot'] and behavior_name == 'network_beaconing':
                detected_behaviors.append({
                    'behavior': behavior_name,
                    'description': behavior_data['description'],
                    'confidence': 0.95,
                    'risk_score': behavior_data['risk_score']
                })
            elif sample['family'] == 'Ryuk' and behavior_name == 'file_encryption':
                detected_behaviors.append({
                    'behavior': behavior_name,
                    'description': behavior_data['description'],
                    'confidence': 0.98,
                    'risk_score': behavior_data['risk_score']
                })
            elif np.random.random() < 0.4:
                detected_behaviors.append({
                    'behavior': behavior_name,
                    'description': behavior_data['description'],
                    'confidence': np.random.uniform(0.6, 0.9),
                    'risk_score': behavior_data['risk_score']
                })

        return detected_behaviors

    def threat_intelligence_lookup(self, sample_hash):
        ti_sources = ['VirusTotal', 'Hybrid Analysis', 'Joe Sandbox', 'Any.run', 'Malwr']

        intel_data = {
            'hash': sample_hash,
            'sources': [],
            'campaigns': [],
            'attribution': None,
            'iocs': []
        }

        for source in random.sample(ti_sources, np.random.randint(2, 5)):
            intel_data['sources'].append({
                'source': source,
                'detection_ratio': f"{np.random.randint(20, 65)}/67",
                'last_analysis': (datetime.now() - timedelta(days=np.random.randint(1, 30))).strftime('%Y-%m-%d')
            })

        campaigns = ['APT29 Campaign', 'Carbanak Group', 'Lazarus Operation', 'FIN7 Activity']
        intel_data['campaigns'] = random.sample(campaigns, np.random.randint(0, 2))

        if np.random.random() < 0.3:
            intel_data['attribution'] = np.random.choice(['APT29', 'Lazarus Group', 'FIN7', 'Carbanak'])

        ioc_types = ['domain', 'ip', 'url', 'email', 'mutex']
        for _ in range(np.random.randint(3, 8)):
            ioc_type = np.random.choice(ioc_types)
            if ioc_type == 'domain':
                intel_data['iocs'].append({'type': 'domain', 'value': f'malicious-{random.randint(1,100)}.com'})
            elif ioc_type == 'ip':
                intel_data['iocs'].append({'type': 'ip', 'value': f'192.168.{random.randint(1,255)}.{random.randint(1,255)}'})

        return intel_data

    def malware_family_analysis(self):
        family_stats = defaultdict(lambda: {'count': 0, 'capabilities': Counter()})

        for sample in self.sample_database:
            family = sample['family']
            family_stats[family]['count'] += 1
            for cap in sample['capabilities']:
                family_stats[family]['capabilities'][cap] += 1

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        families = list(family_stats.keys())
        counts = [family_stats[f]['count'] for f in families]

        axes[0,0].bar(families, counts, color='skyblue')
        axes[0,0].set_title('Malware Family Distribution')
        axes[0,0].set_ylabel('Sample Count')
        axes[0,0].tick_params(axis='x', rotation=45)

        all_capabilities = Counter()
        for family_data in family_stats.values():
            all_capabilities.update(family_data['capabilities'])

        top_caps = dict(all_capabilities.most_common(10))
        axes[0,1].barh(list(top_caps.keys()), list(top_caps.values()), color='lightcoral')
        axes[0,1].set_title('Top Malware Capabilities')
        axes[0,1].set_xlabel('Frequency')

        file_types = Counter(s['file_type'] for s in self.sample_database)
        axes[1,0].pie(file_types.values(), labels=file_types.keys(), autopct='%1.1f%%')
        axes[1,0].set_title('File Type Distribution')

        sizes = [s['size'] for s in self.sample_database]
        axes[1,1].hist(sizes, bins=20, alpha=0.7, color='lightgreen')
        axes[1,1].set_title('File Size Distribution')
        axes[1,1].set_xlabel('Size (bytes)')
        axes[1,1].set_ylabel('Count')

        plt.tight_layout()
        plt.show()

        return dict(family_stats)

    def generate_ioc_report(self):
        all_iocs = {
            'hashes': [],
            'domains': [],
            'ips': [],
            'urls': [],
            'file_paths': []
        }

        for sample in self.sample_database[:20]:
            all_iocs['hashes'].extend([sample['md5'], sample['sha1'], sample['sha256']])

            strings_data = self.extract_strings(sample)
            all_iocs['domains'].extend([url.replace('http://', '').replace('https://', '') for url in strings_data['urls']])
            all_iocs['urls'].extend(strings_data['urls'])

        for _ in range(30):
            all_iocs['ips'].append(f"192.168.{random.randint(1,255)}.{random.randint(1,255)}")
            all_iocs['file_paths'].append(f"C:\\temp\\malware_{random.randint(1,100)}.exe")

        print("🚨 INDICATORS OF COMPROMISE (IOCs)")
        print("=" * 50)

        for ioc_type, values in all_iocs.items():
            if values:
                print(f"\n{ioc_type.upper()}:")
                unique_values = list(set(values))[:10]
                for value in unique_values:
                    print(f"  {value}")

        return all_iocs

    def run_comprehensive_analysis(self):
        print("🔬 MALWARE ANALYSIS LABORATORY")
        print("=" * 50)

        print("\n1. SAMPLE DATABASE OVERVIEW:")
        print(f"Total samples: {len(self.sample_database)}")
        families = Counter(s['family'] for s in self.sample_database)
        print("Top families:")
        for family, count in families.most_common(5):
            print(f"  {family}: {count} samples")

        print("\n2. STATIC ANALYSIS EXAMPLE:")
        sample_id = self.sample_database[0]['id']
        static_result = self.static_analysis(sample_id)
        print(f"Sample: {sample_id}")
        print(f"Family: {static_result['file_info']['family']}")
        print(f"Type: {static_result['file_info']['type']}")
        print(f"Entropy: {static_result['file_info']['entropy']}")
        print(f"YARA matches: {len(static_result['yara_matches'])}")

        print("\n3. DYNAMIC ANALYSIS EXAMPLE:")
        dynamic_result = self.dynamic_analysis(sample_id)
        print(f"Execution time: {dynamic_result['execution_time']}s")
        print(f"Processes spawned: {len(dynamic_result['process_tree'])}")
        print(f"Network connections: {len(dynamic_result['network_activity']['tcp_connections'])}")
        print(f"Behaviors detected: {len(dynamic_result['behavioral_analysis'])}")

        print("\n4. THREAT INTELLIGENCE LOOKUP:")
        sample_hash = self.sample_database[0]['sha256']
        ti_result = self.threat_intelligence_lookup(sample_hash)
        print(f"Hash: {sample_hash[:16]}...")
        print(f"Sources: {len(ti_result['sources'])}")
        print(f"IOCs: {len(ti_result['iocs'])}")
        if ti_result['attribution']:
            print(f"Attribution: {ti_result['attribution']}")

        print("\n5. MALWARE FAMILY ANALYSIS:")
        family_analysis = self.malware_family_analysis()

        print("\n6. IOC GENERATION:")
        ioc_report = self.generate_ioc_report()

        return {
            'static_analysis': static_result,
            'dynamic_analysis': dynamic_result,
            'threat_intelligence': ti_result,
            'family_analysis': family_analysis,
            'iocs': ioc_report
        }

lab = MalwareAnalysisLab()
results = lab.run_comprehensive_analysis()

print(f"\n🎯 INTERACTIVE FEATURES:")
print("Try these commands:")
print("• lab.static_analysis('MAL_0001')")
print("• lab.dynamic_analysis('MAL_0005')")
print("• lab.threat_intelligence_lookup('hash_here')")
print("• lab.malware_family_analysis()")